## MLP & RealMLP: Cross-validation & testing
### Preparing the datasets

In [2]:
from src.data_processor import get_prepared_data, _build_preprocessor, DATASET_CONFIG
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.neural_network import MLPClassifier, MLPRegressor
from scipy.stats import loguniform
import numpy as np
import pandas as pd
from pprint import pprint

# For imbalanced datasets
from imblearn.pipeline import Pipeline as IMLPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# For tuning threshold
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, r2_score, root_mean_squared_error

#### Classification tasks
* For tabular classification and regression tasks, the following hyperparameters would be of interest during the tuning process:
    + `hidden_layer_sizes`: For tabular data, the width of the hidden layers often matters more than the depth. This parameter directly affects the capacity of the model.
    + `alpha`: This parameter controls the strength of the regularization term in the model.
    + `learning_rate_init`: This parameter is only used when the `Adam` or `SGD` solver is used. The cross-validation function tests a number of step sizes for the initial learning rate.
    + `activation`: Controls the activation layer used in the model: Rectified Linear Unit usually works well, but the `tanh` activation layer can help on small tabular problems.
* For classification tasks, highly imbalanced datasets (E.g. The `Classification_n_gt_10k` dataset of credit-card fraud data) could negatively affect model performance. Thus, resampling and tuning the classification threshold could prove beneficial:
    + `imbalance-learn` is used to replace the default Pipeline provided by `scikit-learn`
    + Since both the `Classification_n_gt_10k` and `Classification_n_gt_50` datasets are highly imbalanced, the common strategy of resampling to achieve a $2:1$ ratio of the majority class to the minority class would result in an extremely large dataset
        + Hence, samples from the majority class are dropped until a $10:1$ ratio is achieved. Then, the usual $2:1$ resampling method is applied.
    + After resampling, the threshold for classification is tuned on the validation set using the `F1` score

In [3]:
# Get datasets for classification tasks
datasets_to_run = {
    k: v
    for k, v in DATASET_CONFIG.items()
    if v['task'] == 'classification'
}

# Hyperparameters to test during cross-validation
param_dist = {
    "mlp__hidden_layer_sizes": [(32,), (64,), (64,32), (128,64)],
    "mlp__alpha": loguniform(1e-5, 1e-1),
    "mlp__learning_rate_init": loguniform(1e-4, 1e-2),
    "mlp__activation": ["relu", "tanh"],
}

for dataset_name, config in datasets_to_run.items():
    print(f'Loading and processing: {dataset_name}...')

    try:
        X_train, X_val, X_test, y_train, y_val, y_test = get_prepared_data(dataset_name, return_raw=True)
    except Exception as e:
        print(f'Skipping {dataset_name} due to error: {e}')
        continue

    if dataset_name in ['Classification_n_gt_10k', 'Classification_d_gt_50']:
        pipe = IMLPipeline([
            ('preprocess', _build_preprocessor(X_train)),
            ('under', RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
            ('over', SMOTE(sampling_strategy=0.5, random_state=42)),
            ('mlp', MLPClassifier(max_iter=2000, early_stopping=True, random_state=42)),
        ])

        search = RandomizedSearchCV(
            pipe, param_dist, n_iter=60,
            cv=StratifiedKFold(5, shuffle=True, random_state=42),
            scoring='roc_auc', n_jobs=-1, random_state=42
        )
        print(f'Fitting {dataset_name}...')
        search.fit(X_train, y_train)

        # Print the results of the search
        print(f'Best parameters: ')
        pprint(search.best_params_)
        print(f'Best AUC-ROC: ')
        print(search.best_score_)
        print('')

        # Train a model with the best hyperparameters
        best_pipe = search.best_estimator_
        best_pipe.fit(X_train, y_train)

        # Get prediction probabilities on the validation dataset, only on positive classifications
        print('Finding the threshold...')
        val_prob = best_pipe.predict_proba(X_val)[:, 1]

        # Sweep threshold & find one that maximizes the F1 score
        precision, recall, thresholds = precision_recall_curve(y_val, val_prob)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        best_idx = np.argmax(f1[:-1])
        best_threshold = thresholds[best_idx]

        # Print validation results
        print(f'Best threshold: {best_threshold}')
        print(f'Validation dataset - F1: {f1[best_idx]}')
        print(f'Validation dataset - Precision: {precision[best_idx]}')
        print(f'Validation dataset - Recall: {recall[best_idx]}')

        # Test trained model
        test_prob = best_pipe.predict_proba(X_test)[:, 1]
        test_pred = (test_prob >= best_threshold).astype(int)

        print(f'Test dataset - AUC-ROC: {roc_auc_score(y_test, test_prob)}')
        print(f'Test dataset - F1 @ {best_threshold}: {f1_score(y_test, test_pred)}')
        print('')
    else:
        X_trainval = pd.concat([X_train, X_val], axis=0)
        y_trainval = np.concatenate([y_train, y_val], axis=0)

        pipe = Pipeline([
            ('preprocess', _build_preprocessor(X_trainval)),
            ('mlp', MLPClassifier(max_iter=2000, early_stopping=True, random_state=42)),
        ])

        search = RandomizedSearchCV(
            pipe, param_dist, n_iter=60,
            cv=StratifiedKFold(5, shuffle=True, random_state=42),
            scoring='roc_auc', n_jobs=-1, random_state=42
        )
        print(f'Fitting {dataset_name}...')
        search.fit(X_trainval, y_trainval)

        # Print the results of the search
        print(f'Best parameters: ')
        pprint(search.best_params_)
        print(f'Best AUC-ROC: ')
        print(search.best_score_)
        print('')

        # Test trained model on test dataset
        best_pipe = search.best_estimator_
        best_pipe.fit(X_trainval, y_trainval)
        test_prob = best_pipe.predict_proba(X_test)[:, 1]
        test_pred = best_pipe.predict(X_test)

        # Present test results
        print(f'Test result: AUC-ROC: {roc_auc_score(y_test, test_prob)}')
        print(f'Test result: F1 @ default threshold: {f1_score(y_test, test_pred)}')
        print('')


Loading and processing: Classification_n_gt_10k...
Fitting Classification_n_gt_10k...
Best parameters: 
{'mlp__activation': 'tanh',
 'mlp__alpha': np.float64(2.1057814970278994e-05),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.00029872741995638395)}
Best AUC-ROC: 
0.9820779149460661

Finding the threshold...
Best threshold: 0.99098175729642
Validation dataset - F1: 0.7692307692302698
Validation dataset - Precision: 0.7971014492753623
Validation dataset - Recall: 0.7432432432432432
Test dataset - AUC-ROC: 0.9771552292303558
Test dataset - F1 @ 0.99098175729642: 0.8120300751879699

Loading and processing: Classification_d_gt_50...
Fitting Classification_d_gt_50...
Best parameters: 
{'mlp__activation': 'tanh',
 'mlp__alpha': np.float64(0.008111253665497063),
 'mlp__hidden_layer_sizes': (64, 32),
 'mlp__learning_rate_init': np.float64(0.00015030900645056822)}
Best AUC-ROC: 
0.9257358679484986

Finding the threshold...
Best threshold: 0.8295634304826549
V

#### Regression tasks
* The cross-validation search for regression tasks uses the $R^2$ score to measure performance across folds
    + $R^2$ measures how much of the variance of the predictions $y_i$ can be explained by the variance of the input data $X_i$ for the $i^{\text{th}}$ sample
* Findings based on training & evaluation metrics
    + The `Regression_n_gt_10k` dataset, which contains data about the properties of protein tertiary strcuture, has a high noise floor
        + As a result, the $R^2$ score looked fairly middling
        + This is expected, as most MLP implementations land around a score of $0.55 - 0.65$
    + The `Regression_d_gt_50` dataset, which is a featured-engineered version of the popular AMES housing dataset, has strong features
        + Good models tend to achieve an $R^2$ score of $0.88 - 0.93$
        + As such, the performance achieved by MLP in this case is good
    * The `Regression_Mixed` dataset, which contains data about insurance charges, has few samples (Roughly ~1.3k)
        + This meant that, if the test dataset contains "harder" examples (I.e. Samples with targeted value not in line with pattern learned in training dataset), performance on the test set might be worse than expected
        + This is within expectations for good models on this dataset

In [4]:
# Get datasets for regression tasks
datasets_to_run_reg = {
    k: v
    for k, v in DATASET_CONFIG.items()
    if v['task'] == 'regression'
}

for dataset_name, config in datasets_to_run_reg.items():
    print(f'Loading and processing: {dataset_name}...')
    try:
        X_train, X_val, X_test, y_train, y_val, y_test = get_prepared_data(dataset_name, return_raw=True)
    except Exception as e:
        print(f'Skipping {dataset_name} due to error: {e}')
        continue

    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = np.concatenate([y_train, y_val], axis=0)

    pipe = Pipeline([
        ('preprocess', _build_preprocessor(X_trainval)),
        ('mlp', MLPRegressor(max_iter=2000, early_stopping=True, random_state=42)),
    ])

    search = RandomizedSearchCV(
        pipe, param_dist, n_iter=60,
        cv=KFold(5, shuffle=True, random_state=42),
        scoring='r2', n_jobs=-1, random_state=hash(dataset_name) % (2**32)
    )
    print(f'Fitting {dataset_name}...')
    search.fit(X_trainval, y_trainval)

    print(f'Best parameters: ')
    pprint(search.best_params_)
    print(f'Best R squared: ')
    print(search.best_score_)
    print('')

    # Fitting the model with best hyperparameters
    best_pipe = search.best_estimator_
    best_pipe.fit(X_trainval, y_trainval)

    # Testing the model
    y_pred = best_pipe.predict(X_test)
    print(f'Test result: R squared: {r2_score(y_test, y_pred)}')
    print(f'Test result: RSME: {root_mean_squared_error(y_test, y_pred)}')
    print('')

Loading and processing: Regression_n_gt_10k...
Fitting Regression_n_gt_10k...
Best parameters: 
{'mlp__activation': 'tanh',
 'mlp__alpha': np.float64(0.08591342830048515),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.00615533906228275)}
Best R squared: 
0.574267962335696

Test result: R squared: 0.584955059437027
Test result: RSME: 3.973914541484349

Loading and processing: Regression_d_gt_50...
Fitting Regression_d_gt_50...


/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

Best parameters: 
{'mlp__activation': 'relu',
 'mlp__alpha': np.float64(0.07423121900844358),
 'mlp__hidden_layer_sizes': (64, 32),
 'mlp__learning_rate_init': np.float64(0.006741738304421186)}
Best R squared: 
0.8341743643302031

Test result: R squared: 0.8525888554284531
Test result: RSME: 32597.164379109592

Loading and processing: Regression_Mixed...
Fitting Regression_Mixed...


/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

Best parameters: 
{'mlp__activation': 'relu',
 'mlp__alpha': np.float64(0.0066157058689567585),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.009776977915629067)}
Best R squared: 
0.8244958827724439

Test result: R squared: 0.8623846944056957
Test result: RSME: 4568.287189026236

